# I.P. for ResNet-34 / CIFAR-100

The last gap in the grid. Thunder holds the CIFAR-100 BaCP runs; this is the
matched I.P. arm, 50 pruning epochs plus the same 25-epoch mask-frozen
fine-tune at SGD 0.01.

**These resume rather than rerun.** Each cell's 50-epoch pruning run already
happened and saved its sparse checkpoint to DBFS, so `--finetune_only` reads the
mask back off the zeros and runs only the fine-tune. 36 cells, ~0.8 h.

The ResNet CIFAR-10 `.ft25` cells ran end-to-end while these resume; the
appendix should say so.


In [ ]:
import sys, pathlib

here = pathlib.Path.cwd()
while not (here / '.git').exists() and here != here.parent:
    here = here.parent
sys.path.insert(0, str(here / 'project' / 'test_notebooks'))

import nb_common as nb
info = nb.setup()


## Plan

In [ ]:
import json, glob, os, re

MODEL, DATASET = 'resnet34', 'cifar100'
PRUNERS    = ('magnitude', 'snip', 'wanda')
SPARSITIES = (0.95, 0.97, 0.99, 0.999)
SEEDS      = (1, 2, 3)
GPU        = 0

FT = dict(enable_finetune=True, epochs_ft=25,
          optimizer_type_ft='adamw', learning_rate_ft=1e-4,
          finetune_only=True)

# sparsity carries a dot, so the key is matched with an anchored regex
KEY = re.compile(r'^static\.prune\.resnet34\.cifar100\.'
                 r's(0\.\d+)\.(magnitude|snip|wanda)\.seed(\d+)$')

root = os.environ['BACP_RESULTS_DIR']
sparse, done = {}, set()
bad = 0
for f in glob.glob(os.path.join(root, 'runs', '*.json')):
    try:
        r = json.load(open(f, encoding='utf-8'))
    except Exception:
        bad += 1
        continue
    g = r.get('experiment_group') or ''
    if r.get('status') != 'ok':
        continue
    m = KEY.match(g)
    if m and r.get('save_path'):
        sparse[(m.group(1), m.group(2), int(m.group(3)))] = r['save_path']
    if g.endswith('.ft25') and '.resnet34.cifar100.' in g:
        done.add(g)

print('sparse checkpoints found: %d (%d unreadable records skipped)' % (len(sparse), bad))

plan, absent = [], []
for p in PRUNERS:
    for sp in SPARSITIES:
        for s in SEEDS:
            key = 'static.prune.%s.%s.s%s.%s.seed%d.ft25' % (MODEL, DATASET, sp, p, s)
            if key in done:
                continue
            ck = sparse.get((str(sp), p, s))
            if ck is None or not os.path.exists(ck):
                absent.append((sp, p, s, ck or '(no record)'))
                continue
            plan.append(nb.make_cell(MODEL, 'prune', seed=s, pruner=p, sparsity=sp,
                                     variant='ft25', trained_weights=ck,
                                     dataset_name=DATASET, num_classes=100, **FT))

if absent:
    print('\nMISSING %d checkpoint(s), those cells are skipped:' % len(absent))
    for a in absent:
        print('   ', a)

ref = nb.FAMILIES[MODEL]['prune']
for c in plan:
    for k in ('learning_rate', 'epochs', 'delta_T', 'sparsity_scheduler',
              'recovery_epochs', 'val_split', 'prune_task_head', 'wanda_group',
              'optimizer_type', 'batch_size'):
        if k in ref:
            assert c['config'][k] == ref[k], (c['key'], k, c['config'][k], ref[k])
    assert c['config']['learning_rate'] == 0.01
    assert c['config']['epochs'] == 50 and c['config']['epochs_ft'] == 25
    assert c['config']['num_classes'] == 100
    assert c['config']['finetune_only'] is True
    assert c['key'].endswith('.ft25')
    assert 'static-prune' in c['config']['trained_weights'], c['config']['trained_weights']

assert len({c['key'] for c in plan}) == len(plan), 'duplicate key'
print('\n%d cells, ~%.1f h (25 epochs each, resumed)' % (len(plan), len(plan) * 1.34 / 60))
assert nb.sanity_check(plan), 'sanity check failed'


## Run

In [ ]:
nb.run_group(plan, gpu=GPU)


## Results

In [ ]:
import json, glob, os, statistics as st

root = os.environ['BACP_RESULTS_DIR']
acc = {}
for f in glob.glob(os.path.join(root, 'runs', '*.json')):
    try:
        r = json.load(open(f, encoding='utf-8'))
    except Exception:
        continue
    k = r.get('experiment_group') or ''
    if r.get('status') == 'ok' and '.smoke' not in k:
        acc[k] = r.get('test_acc_exact_pct') or r.get('test_acc_pct')

def cell(arm, sp, p, suffix=''):
    xs = [acc.get('static.%s.%s.%s.s%s.%s.seed%d%s' % (arm, MODEL, DATASET, sp, p, s, suffix))
          for s in SEEDS]
    xs = [x for x in xs if x is not None]
    return (st.mean(xs), len(xs)) if xs else None

print('%-22s %9s %9s %9s' % ('cell', 'I.P.', 'BaCP', 'delta'))
print('-' * 54)
for p in PRUNERS:
    for sp in SPARSITIES:
        ip, bp = cell('prune', sp, p, '.ft25'), cell('bacp', sp, p)
        f = lambda v: '   --  ' if v is None else '%6.2f%s' % (v[0], '*' if v[1] < 3 else ' ')
        d = '%+8.2f' % (bp[0] - ip[0]) if ip and bp else '    --  '
        print('%-22s %9s %9s %9s' % ('%s %s' % (p, sp), f(ip), f(bp), d))
print()
print('* = fewer than 3 seeds')
